# Did the away side create better chances?

The two teams were close to identical on process stats: shots, passes, pass completion, distance covered. But away won by two goals.
This notebook tests one explanation: **away's shots were higher quality**. It checks where shots were taken from (event data) and how much pressure the shooter was under (tracking data).

Run top to bottom. It needs the sprint dataset in `../data/` (not included in this repo; see the README).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pitch import PITCH_L, COLOURS, load, Frames, draw_pitch, plot_frame, video_time

events, players = load()
frames = Frames(players)
print(f"events {len(events):,} | tracking rows {len(players):,} | frames {len(frames.times):,}")

## 1. Where were the shots taken from?
Median distance and angle to goal, and the share taken inside the penalty box (16.5 m deep, 40.3 m wide).

In [ ]:
shots = events[events.event_name == "shot"].copy()
shots["in_box"] = (shots.x_att > PITCH_L / 2 - 16.5) & (shots.y_att.abs() < 20.15)

shots.groupby("team").agg(
    shots=("event_id", "size"),
    median_dist_m=("shot_distance_m", "median"),
    median_angle_deg=("shot_angle_deg", "median"),
    share_in_box=("in_box", "mean"),
    scored=("is_goal", "sum"),
).round(2)

## 2. Every shot on one pitch
`x_att` / `y_att` flip both halves so every team attacks left → right. Stars are goals.

In [ ]:
ax = draw_pitch()
for team, g in shots.groupby("team"):
    ax.scatter(g.x_att, g.y_att, s=80, c=COLOURS[team], alpha=0.8, label=team, zorder=3)
    goals = g[g.is_goal]
    ax.scatter(goals.x_att, goals.y_att, s=260, marker="*", c=COLOURS[team], edgecolor="black", zorder=4)
ax.set_xlim(0, PITCH_L / 2 + 3)          # attacking half only
ax.legend()
ax.set_title("Shots (stars = goals), attacking left → right")
plt.show()

## 3. How much pressure was the shooter under?
At the tracking frame nearest each shot:
- **nearest_def_m**: distance to the closest opponent
- **defenders_in_cone**: opponents inside the triangle from the shooter to the two posts (+0.5 m for body width). The goalkeeper counts, so about 1 is normal.

In [ ]:
GOAL_X, POST_Y, BODY_M = PITCH_L / 2, 3.66, 0.5

def pressure_at_shot(shot):
    f = frames.at(shot.timestamp_ms)
    opp = f[(f.team != shot.team) & (f.team != "referee")]
    ox, oy = opp.x * shot.attack_dir, opp.y * shot.attack_dir   # same frame as the shooter
    sx, sy = shot.x_att, shot.y_att

    nearest = np.hypot(ox - sx, oy - sy).min()

    frac = (ox - sx) / (GOAL_X - sx)          # 0 at the shooter, 1 on the goal line
    centre = sy * (1 - frac)                  # cone centre line, shooter -> middle of goal
    in_cone = (ox > sx) & (ox <= GOAL_X) & ((oy - centre).abs() <= POST_Y * frac + BODY_M)
    return pd.Series({"nearest_def_m": nearest, "defenders_in_cone": in_cone.sum()})

shots = shots.join(shots.apply(pressure_at_shot, axis=1))
shots.groupby("team")[["nearest_def_m", "defenders_in_cone"]].mean().round(2)

## 4. Freeze-frame of a goal
Change `GOAL` to look at the others, then find the moment in the video with `video_time`.

In [ ]:
GOAL = 0
s = shots[shots.is_goal].iloc[GOAL]
plot_frame(frames.at(s.timestamp_ms), title=f"{s.player_id} shot, video {video_time(s.timestamp_ms)}")
plt.show()

## Takeaways
- **Location:** away shot from much closer to goal and from inside the box far more often, on a near-equal number of shots.
- **Pressure:** away's shooters also had more space, with the nearest defender further away and fewer bodies between ball and goal.
- So the two sides took the same *number* of shots, but away took *better* ones on both counts.
- Caveat: about 30 shots in one match describe one game, not a model. A fitted xG needs many matches.